#🧩 Step 1: Install Required Libraries for RAG Implementation

In this step, we install all essential libraries for building a Retrieval-Augmented Generation (RAG) system using LangChain, OpenAI, and Pinecone.
These packages enable document loading, text chunking, embedding generation, vector storage, and LLM-based retrieval.


In [1]:
# 🧩 Step 1: Install Required Libraries for RAG Implementation

# This command installs and updates all core dependencies needed
# to build a complete RAG pipeline with LangChain, OpenAI, and Pinecone.
!pip -q install -U \
  langchain langchain-community==0.2.* langchain-openai \
  langchain-pinecone==0.1.* pinecone==4.* pypdf==4.* tiktoken


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.4/214.4 kB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.8/295.8 kB 20.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.5/51.5 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 397.1/397.1 kB 24.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 311.8/311.8 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 85.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 244.8/244.8 kB 19.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#🔐 Step 2: Configure Environment Variables and API Keys

In this step, we securely configure all environment variables and API keys required for our LangChain + Pinecone RAG pipeline.
Using Colab’s userdata.get() ensures API keys remain private and are not exposed directly in the notebook.

In [ ]:
# 🔐 Step 2: Configure Environment Variables and API Keys

from google.colab import userdata
import os

# Enable LangSmith tracing for tracking and visualizing LangChain runs
os.environ["LANGSMITH_TRACING"] = "true"

# Define LangSmith endpoint and authentication key
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign project name to group all related traces under one dashboard
os.environ["LANGSMITH_PROJECT"] = "pr-abandoned-canvas-88"

# Retrieve API keys securely from Colab's secret manager
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["PINECONE_API_KEY"] = userdata.get('PINECONE_API_KEY')

# Embedding model: 1536-dimensional OpenAI model for text representations
OPENAI_MODEL_EMB = "text-embedding-3-small"

# Pinecone index configuration
INDEX_NAME = "day3-rag-index"     # Index name for storing document embeddings
PINECONE_CLOUD = "aws"            # Cloud provider (can also be "gcp")
PINECONE_REGION = "us-east-1"     # Region where Pinecone index is hosted


#🧠 Step 3: Import Core Libraries for the RAG Pipeline

In this step, we import all the essential libraries needed for document processing, embedding generation, and vector database integration.
These modules work together to load, split, and store documents in Pinecone for efficient retrieval during question-answering.

In [ ]:
# 🧠 Step 3: Import Core Libraries for the RAG Pipeline

# Standard utilities for file paths, metadata, and typing
from pathlib import Path          # To manage dataset folder paths
import json                       # For reading and writing metadata
from typing import List            # For type hinting lists of documents

# LangChain components for OpenAI models
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # Embeddings + Chat models

# Document loaders for PDF and text file ingestion
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Text splitter to divide documents into small chunks suitable for embeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Document schema defines how text + metadata are stored in LangChain
from langchain.schema import Document

# Pinecone SDK and LangChain vector store integration
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore


#📂 Step 4: Load and Structure Core and Student Documents

In this step, we load all training and reference documents for the RAG pipeline.
The function reads both core PDFs (common reference materials) and student-specific folders that contain reports, summaries, and metadata.
Each file is converted into a LangChain Document object with clear metadata tags to enable precise filtering and retrieval later.

In [ ]:
# 📂 Step 4: Load and Structure Core and Student Documents

# Define the main directory containing the training data
# Make sure your Google Drive is mounted and the path exists
DATA = Path("/content/drive/MyDrive/APU TRAINING/TRAINING DATA")

# List of common PDFs stored at the root of the data folder
CORE_PDFS = [
    DATA / "projects.pdf",
    DATA / "criteria.pdf",
    DATA / "mentors.pdf",
]

def load_core_pdfs() -> List[Document]:
    """
    Loads core documents such as projects, criteria, and mentors.
    Each page is parsed as a LangChain Document with category metadata.
    """
    docs = []
    for pdf in CORE_PDFS:
        if pdf.exists():
            # Load all pages of the PDF as Document objects
            pages = PyPDFLoader(str(pdf)).load()
            # Add consistent metadata for easier retrieval
            for p in pages:
                p.metadata.update({"source": pdf.name, "category": "core"})
            docs.extend(pages)
    return docs


def load_student_dirs() -> List[Document]:
    """
    Loads documents from each student folder, including:
    - report.pdf
    - summary.txt (optional)
    - metadata.json (optional)
    Each is tagged with the student's name and file category.
    """
    docs = []
    students_dir = DATA / "students"
    if not students_dir.exists():
        return docs

    for student_dir in students_dir.iterdir():
        if not student_dir.is_dir():
            continue

        # Load student report (PDF)
        report_pdf = student_dir / "report.pdf"
        if report_pdf.exists():
            pages = PyPDFLoader(str(report_pdf)).load()
            for p in pages:
                p.metadata.update({
                    "source": f"{student_dir.name}/report.pdf",
                    "student": student_dir.name,
                    "category": "student_report"
                })
            docs.extend(pages)

        # Load student summary (TXT)
        summary_txt = student_dir / "summary.txt"
        if summary_txt.exists():
            tdocs = TextLoader(str(summary_txt), encoding="utf-8").load()
            for d in tdocs:
                d.metadata.update({
                    "source": f"{student_dir.name}/summary.txt",
                    "student": student_dir.name,
                    "category": "student_summary"
                })
            docs.extend(tdocs)

        # Load student metadata (JSON)
        meta_json = student_dir / "metadata.json"
        if meta_json.exists():
            try:
                meta = json.loads(meta_json.read_text(encoding="utf-8"))
                meta_doc = Document(
                    page_content=json.dumps(meta, ensure_ascii=False, indent=2),
                    metadata={
                        "source": f"{student_dir.name}/metadata.json",
                        "student": student_dir.name,
                        "category": "student_metadata"
                    }
                )
                docs.append(meta_doc)
            except Exception as e:
                print(f"Could not parse {meta_json}: {e}")

    return docs


# Combine all loaded documents
raw_docs = load_core_pdfs() + load_student_dirs()

# Print total count of loaded documents
print(f"Loaded {len(raw_docs)} raw documents (pages + text).")


Loaded 131 raw documents (pages + text).


#✂️ Step 5: Split Documents into Smaller, Contextual Chunks

In this step, we divide long documents into smaller, overlapping chunks to make them more suitable for embedding and retrieval.
This improves both semantic search accuracy and context retention when the model answers questions later.

Key parameters:

chunk_size=1000 → Maximum number of characters per chunk.

chunk_overlap=150 → Overlap between chunks to preserve continuity.

separators → Defines the order in which text is split (paragraphs → lines → spaces → characters).

After splitting, we print the total number of chunks created and preview the first one to verify formatting and metadata.

In [ ]:
# ✂️ Step 5: Split Documents into Smaller, Contextual Chunks

# Initialize the RecursiveCharacterTextSplitter to divide long text into smaller sections.
# It recursively splits by paragraphs, then lines, then words, then characters.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,                # Max characters per chunk
    chunk_overlap=150,              # Overlap between consecutive chunks for better context continuity
    separators=["\n\n", "\n", " ", ""],  # Preferred order of splitting (logical to granular)
)

# Apply the splitter to all loaded raw documents
chunks = splitter.split_documents(raw_docs)

# Display the total number of generated text chunks
print(f"Created {len(chunks)} chunks.")

# Print the first 300 characters of the first chunk to verify
print(chunks[0].page_content[:300], "...\n", chunks[0].metadata)


Created 282 chunks.
PROJECT
 
EVALUATION
 
CRITERIA
 
2024
 
 
GRADING
 
RUBRIC
 
 
TECHNICAL
 
EXCELLENCE
 
(40
 
points)
 
-
 
Code
 
Quality
 
and
 
Documentation:
 
10
 
pts
 
-
 
Algorithm
 
Efficiency:
 
10
 
pts
 
-
 
System
 
Architecture:
 
10
 
pts
 
-
 
Testing
 
and
 
Validation:
 
10
 
pts
 
 
INNOVATION
  ...
 {'source': 'criteria.pdf', 'page': 0, 'category': 'core'}


#Step 6: Generate Embeddings and Store Them in Pinecone

In this step, we convert each text chunk into vector embeddings using OpenAI’s embedding model and store them in a Pinecone vector database.
This forms the semantic backbone of the RAG system — allowing the model to later retrieve contextually relevant text when answering questions.

Process Overview:

Embedding generation → Converts text chunks into high-dimensional numeric vectors (1536 dimensions for text-embedding-3-small).

Pinecone index setup → Creates or connects to a vector index where embeddings will be stored.

Vector store creation → Wraps the index with LangChain’s PineconeVectorStore for easy document retrieval.

In [ ]:
# 🧮 Step 6: Generate Embeddings and Store Them in Pinecone

# Initialize the OpenAI embeddings model
# Each chunk will be converted into a 1536-dimensional vector representation
embeddings = OpenAIEmbeddings(model=OPENAI_MODEL_EMB)

# Initialize Pinecone client with your API key
pc = Pinecone(api_key=os.environ["PINECONE_API_KEY"])

# Create a new index if it doesn't already exist in your Pinecone workspace
# The dimension must match the embedding model output (1536 for text-embedding-3-small)
if INDEX_NAME not in [ix["name"] for ix in pc.list_indexes()]:
    pc.create_index(
        name=INDEX_NAME,                 # Name of the Pinecone index
        dimension=1536,                  # Must match embedding model dimensions
        metric="cosine",                 # Similarity metric for semantic matching
        spec=ServerlessSpec(
            cloud=PINECONE_CLOUD,        # Cloud provider (e.g., AWS, GCP)
            region=PINECONE_REGION       # Geographic region for your index
        ),
    )

# Create a LangChain vector store to manage document embeddings in Pinecone
# This step uploads all document chunks as vectorized data for retrieval
vectorstore = PineconeVectorStore.from_documents(
    documents=chunks,                    # List of document chunks to embed
    embedding=embeddings,                # Embedding model instance
    index_name=INDEX_NAME,               # Pinecone index name
)


# If ingesion already done then do this instead

In [2]:
from google.colab import userdata
import os

# Standard utilities
from pathlib import Path            # For handling file and folder paths
import json                         # For reading and writing metadata
from typing import List             # For adding type hints (e.g., List[Document])

# LangChain integrations for OpenAI
from langchain_openai import OpenAIEmbeddings, ChatOpenAI  # For embeddings and chat models

# Loaders to extract content from different document formats
from langchain_community.document_loaders import PyPDFLoader, TextLoader

# Tool to split long documents into manageable chunks for embedding
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Core LangChain document representation (content + metadata)
from langchain.schema import Document

# Pinecone SDK and LangChain vector store wrapper
from pinecone import Pinecone, ServerlessSpec
from langchain_pinecone import PineconeVectorStore

# Enable LangSmith tracing for detailed observability of LangChain runs.
os.environ["LANGSMITH_TRACING"] = "true"

# Define the LangSmith API endpoint.
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"

# Retrieve the LangSmith API key securely from Colab's secret store.
os.environ["LANGSMITH_API_KEY"] = userdata.get('LANGSMITH_API_KEY')

# Assign a project name to organize runs within LangSmith.
os.environ["LANGSMITH_PROJECT"] = "RAG PROJECT"

# 2) Connect to existing index
INDEX_NAME = "day3-rag-index"   # <-- put your existing index name here
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

pc = Pinecone(api_key=userdata.get('PINECONE_API_KEY'))
index = pc.Index(INDEX_NAME)

# 3) Attach embeddings + vectorstore wrapper (NO re-indexing happens here)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = PineconeVectorStore(index=index, embedding=embeddings)


#💬 Step 7: Initialize the Language Model and Retriever

In this step, we set up the language model (LLM) and the retriever that will fetch relevant document chunks from the vector store.
The retriever acts as the bridge between the user’s query and the stored embeddings, ensuring only the most contextually relevant chunks are passed to the model.

Key Details:

ChatOpenAI → Provides access to OpenAI chat models (here, a lightweight gpt-4o-mini model is used for cost efficiency).

temperature=0 → Ensures deterministic and consistent outputs — ideal for RAG tasks.

retriever → Queries the Pinecone vector store and returns the top k most relevant chunks (k=4 here, tunable based on context size).

In [3]:
# 💬 Step 7: Initialize the Language Model and Retriever

# Initialize the OpenAI chat model
# 'gpt-4o-mini' is fast and affordable, suitable for RAG demos
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)  # Set temperature to 0 for factual, repeatable results

# Create a retriever from the Pinecone vector store
# The retriever fetches the top 'k' most similar chunks for a given user query
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


#🧩 Step 8: Implement Query Decomposition for Smarter Retrieval

In this step, we create a Query Decomposition Chain — a preprocessing layer that breaks a complex user question into smaller, more specific factual sub-queries.
This improves retrieval accuracy in RAG pipelines by targeting multiple aspects of the question (like skills, domain, or experience) instead of relying on a single broad query.

In [4]:
# 🧩 Step 8: Implement Query Decomposition for Smarter Retrieval

# Import necessary components for building the decomposition chain
from langchain.prompts import ChatPromptTemplate, PromptTemplate
from langchain.chains import LLMChain
from langchain.schema.runnable import RunnablePassthrough
from langchain.schema import StrOutputParser

# Define a specialized prompt template to instruct the model
# to break a complex user query into multiple factual sub-queries
template = """You are a Query Decomposition Assistant.

Break the input question into 3–5 short factual sub-queries that will retrieve the
specific attributes implied by the main question such as: skills, experience, field of work,
domain knowledge, or relevant achievements.

Rules:
• Identify the core entities and the implied requirements in the question.
• Each sub-query must target one required attribute (e.g., skill, domain, experience).
• No generic questions (e.g., “What is the focus of the project?”).
• No procedural or meta-queries (e.g., “What are the criteria?”).
• No answering the question.

Input: {question}

Output:"""

# Create a ChatPromptTemplate from the defined template
prompt_decomposition = ChatPromptTemplate.from_template(template)

# Initialize a lightweight OpenAI chat model (deterministic and cost-efficient)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Chain the prompt → LLM → output parser → string-to-list conversion
generate_queries_decomposition = (
    prompt_decomposition
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))  # Convert multiline string to list of queries
)

# Example complex user question
question = "Provide me the names of two students for upcoming project that uses robot for teaching the students"

# Invoke the chain to generate decomposed sub-queries
questions = generate_queries_decomposition.invoke({"question": question})


In [5]:
questions

['1. What are the names of students with experience in robotics?',
 '2. Which students have a background in education or teaching methodologies?',
 '3. Who are the students currently enrolled in relevant engineering or technology courses?']

#🧱 Step 9: Create a Composite Prompt for Answer Generation

In this step, we define a structured prompt template that combines three information layers:

Main question → The user’s current query.

Background Q&A pairs → Any previously answered or related questions.

Context → Retrieved chunks or supporting documents from the vector store.

This composite prompt ensures the LLM has access to relevant background knowledge and contextual grounding before generating an answer.
It is a key element in multi-turn or context-aware RAG systems.

In [6]:
# 🧱 Step 9: Create a Composite Prompt for Answer Generation

# Define a structured prompt template that provides:
# 1. The current user question
# 2. Any available background question-answer pairs
# 3. Relevant context from document retrieval
template = """Here is the question you need to answer:

\n --- \n {question} \n --- \n

Here is any available background question + answer pairs:

\n --- \n {q_a_pairs} \n --- \n

Here is additional context relevant to the question:

\n --- \n {context} \n --- \n

Use the above context and any background question + answer pairs to answer the question: \n {question}
"""

# Convert the raw template into a LangChain ChatPromptTemplate
# so it can be used seamlessly with chains and LLMs
decomposition_prompt = ChatPromptTemplate.from_template(template)


#🧠 Step 10: Build and Execute the Multi-Stage RAG Chain (Decomposition + Context Synthesis)

In this step, we connect everything into a two-stage reasoning pipeline that first answers the decomposed sub-queries and then synthesizes a final, context-aware answer.
The pipeline leverages retrieved document chunks, previously answered Q&A pairs, and the composite prompt created earlier.

Workflow Overview:

Format helper (format_qa_pair) → Formats each question–answer pair neatly for reuse in later prompts.

Per-subquery chain (rag_chain) →

Retrieves context for each sub-question.

Uses the LLM to produce a factual answer.

Appends the result to the growing list of Q&A pairs.

Final synthesis chain (final_rag_chain) →

Uses the original complex question plus all gathered Q&A pairs.

Retrieves global context again and asks the LLM to generate a consolidated final answer.

This design allows the model to reason over structured intermediate knowledge before forming the final output—improving precision and reducing hallucinations.

In [7]:
# 🧠 Step 10: Build and Execute the Multi-Stage RAG Chain (Decomposition + Context Synthesis)

from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

def format_qa_pair(question, answer):
    """Format a question-answer pair into a readable text block."""
    formatted_string = ""
    formatted_string += f"Question: {question}\nAnswer: {answer}\n\n"
    return formatted_string.strip()

# Initialize a higher-capacity LLM for synthesis (can use GPT-4o for richer reasoning)
llm = ChatOpenAI(model_name="gpt-4o", temperature=0)

# Initialize an empty string to accumulate Q&A pairs
q_a_pairs = ""

# Iterate through each decomposed sub-query
for q in questions:

    # Define a mini-RAG chain to answer each sub-question independently
    rag_chain = (
        {
            "context": itemgetter("question") | retriever,  # Retrieve context relevant to this sub-question
            "question": itemgetter("question"),             # Current sub-question
            "q_a_pairs": itemgetter("q_a_pairs")            # Previous Q&A context (if any)
        }
        | decomposition_prompt                              # Use composite prompt structure
        | llm                                               # Generate sub-answer
        | StrOutputParser()                                 # Convert LLM output to plain string
    )

    # Invoke the chain for the current sub-question
    answer = rag_chain.invoke({"question": q, "q_a_pairs": q_a_pairs})

    # Format and append the result for later synthesis
    q_a_pair = format_qa_pair(q, answer)
    q_a_pairs = q_a_pairs + "\n---\n" + q_a_pair


# 🔄 Final RAG chain to synthesize the comprehensive answer
final_rag_chain = (
    {
        "context": itemgetter("original_question") | retriever,  # Retrieve global context
        "question": itemgetter("original_question"),              # Original complex question
        "q_a_pairs": itemgetter("q_a_pairs")                      # All sub-query Q&A pairs
    }
    | decomposition_prompt                                        # Same structured prompt
    | llm                                                         # Synthesize final response
    | StrOutputParser()                                           # Parse to text
)

# Run the final synthesis stage
final_answer = final_rag_chain.invoke(
    {"original_question": question, "q_a_pairs": q_a_pairs}
)


In [8]:
print(final_answer)

Based on the provided context, the two students suitable for an upcoming project that uses robots for teaching students are:

1. Kenji Tanaka - He has experience in robotics, as indicated by his project on an autonomous delivery robot with obstacle avoidance.

2. Lisa - She has a background in education or teaching methodologies, which would be beneficial for a project focused on teaching students using robots.
